# 02 — DoE Comparison (Phase 1)

Compares 4 sampling strategies (Random, LHS, Maximin LHS, Sobol) across 7 sample sizes.

**Experiments:** DoE-1 (fixed budget), DoE-2 (seed sensitivity)

**Source:** `outputs/phase1/doe1_summary.csv`, `doe2_summary.csv`, `decision_gate.txt`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
P1 = ROOT / 'outputs' / 'phase1'

## DoE-1: RMSE vs Sample Size

In [ ]:
doe1 = pd.read_csv(P1 / 'doe1_summary.csv')

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
outputs = ['v_out_mean', 'i_l_mean', 'v_out_ripple', 'i_l_ripple', 'efficiency']
colors = {'random': 'C0', 'lhs': 'C1', 'maximin_lhs': 'C2', 'sobol': 'C3'}

for i, out in enumerate(outputs):
    ax = axes.flat[i]
    sub = doe1[doe1['output'] == out]
    for strat in ['random', 'lhs', 'maximin_lhs', 'sobol']:
        s = sub[sub['strategy'] == strat].sort_values('N')
        ax.plot(s['N'], s['rmse_mean'], '-o', label=strat, color=colors[strat], markersize=4)
        if s['rmse_std'].notna().any():
            ax.fill_between(s['N'], s['rmse_mean'] - s['rmse_std'], 
                          s['rmse_mean'] + s['rmse_std'], alpha=0.15, color=colors[strat])
    ax.set_title(out)
    ax.set_xlabel('N')
    ax.set_ylabel('RMSE')
    ax.grid(True, alpha=0.3)
    if i == 0: ax.legend(fontsize=8)
axes.flat[-1].set_visible(False)
fig.suptitle('DoE-1: RMSE vs Sample Size (all outputs)', fontsize=13)
plt.tight_layout()
plt.show()

## DoE-2: Seed Sensitivity (N=100, 30 seeds)

In [ ]:
doe2 = pd.read_csv(P1 / 'doe2_summary.csv')
vout2 = doe2[doe2['output'] == 'v_out_mean']
print('DoE-2 — Seed sensitivity (v_out_mean, N=100):')
print(vout2[['strategy', 'rmse_mean', 'rmse_std', 'rmse_min', 'rmse_max']].to_string(index=False))

In [ ]:
print(open(P1 / 'decision_gate.txt').read())

## Conclusion

**Decision:** Maximin LHS selected (score 1.194 vs LHS 1.216, Random 1.300).
Provides best space-filling properties at larger sample sizes with similar seed robustness to basic LHS.